# System Basic Configuration

# 系统基础配置


## 步骤1: 创建 home 目录软链接


In [ ]:
%%bash
echo "home link make..."
d="/data/.home"
while IFS= read -r i; do
	rm -fr "$HOME/$(basename "$i")"
	ln -sf "$i" "$HOME/$(basename "$i")"
done < <(find "$d" -mindepth 1 -maxdepth 1 -type d)


## 步骤2: 清理 pacman 锁文件


In [ ]:
%%bash
sudo fuser -k /var/lib/pacman/db.lck 2>/dev/null || true
sudo rm -f /var/lib/pacman/db.lck


## 步骤3: 配置中国镜像源


In [ ]:
%%bash
echo "change pacman mirror..."
sudo tee /etc/pacman.d/mirrorlist >/dev/null <<'EOF'
# # China mirrors
Server = https://mirrors.tuna.tsinghua.edu.cn/manjaro/stable/$repo/$arch
EOF


## 步骤4: 安装 yay 和配置


In [ ]:
%%bash
echo "installing yay..."
sudo pacman -Sy --needed --noconfirm base-devel yay >/dev/null


### 配置 yay


In [1]:
%%bash
dir="$HOME/.config/yay"
mkdir -p "$dir"
tee "$dir/config.json" > /dev/null << 'EOF'
{
    "editor": "nano",
    "pacmanbin": "pacman",
    "pacmanconf": "/etc/pacman.conf",
    "answerclean": "All",
    "removemake": "ask",
    "maxconcurrentdownloads": 5,
    "cleanAfter": false,
    "batchinstall": true,
    "DevelCheckUpdate": false
}
EOF
echo "yay config.json created successfully."


yay config.json created successfully.


### 配置 archlinuxcn 源


In [2]:
%%bash
PACMAN_FILE=/etc/pacman.conf
Server="https://mirrors.tuna.tsinghua.edu.cn/archlinuxcn/\$arch"

if ! grep -q "^\[archlinuxcn\]" "$PACMAN_FILE"; then
	echo -e "\n[archlinuxcn]\nServer = $Server" | sudo tee -a "$PACMAN_FILE" >/dev/null
else
	sudo sed -i "/^\[archlinuxcn\]/,/^\[/{/^Server/d}" "$PACMAN_FILE"
	sudo sed -i "/^\[archlinuxcn\]/a Server = $Server" "$PACMAN_FILE"
fi

echo "✓ archlinuxcn configured: $Server"


✓ archlinuxcn configured: https://mirrors.tuna.tsinghua.edu.cn/archlinuxcn/$arch


### 配置 pacman 选项


In [3]:
%%bash
PACMAN_FILE=/etc/pacman.conf
options=("Color" "ILoveCandy" "ParallelDownloads = 5")

for opt in "${options[@]}"; do
	key=${opt%% *}
	sudo sed -i "/^#\?$key/d; /^\[options\]/a $opt" "$PACMAN_FILE"
done

echo "✓ pacman options configured"


✓ pacman options configured


## 步骤5: 系统更新


In [4]:
%%bash
echo "update system..."
sudo pacman -Syyu --noconfirm >/dev/null
yay -Syyu --noconfirm >/dev/null


update system...


## 步骤6: 安装开发工具


### 配置 GPG


In [ ]:
%%bash
name="kefu"
email="kefu1820@gmail.com"
gd="${GPG_DIR:-$HOME/.gnupg}"

if gpg --list-keys "$email" &>/dev/null; then
	echo "✓ GPG key already exists for $email"
else
	read -sp "Enter GPG passphrase: " pswd
	echo
	
	chmod 700 "$gd"
	
	tmp=$(mktemp)
	cat >"$tmp" <<EOF
%echo Generating GPG key
Key-Type: RSA
Key-Length: 4096
Subkey-Type: RSA
Subkey-Length: 4096
Name-Real: $name
Name-Email: $email
Expire-Date: 3y
Passphrase: $pswd
%commit
%echo done
EOF
	
	gpg --batch --generate-key "$tmp"
	rm -f "$tmp"
	
	find "$gd" -type f -exec chmod 600 {} \;
	
	echo "✓ GPG key generated for $email"
	gpg --list-keys "$email"
fi


### 安装 VSCode


In [5]:
%%bash
yay -S --needed --noconfirm visual-studio-code-bin >/dev/null


## 步骤7: 安装 jupyter


In [6]:
%%bash
echo "installing jupyter..."
sudo pacman -S --needed --noconfirm python-pip python-ipykernel jupyter-notebook >/dev/null

f=$HOME/.jupyter/jupyter_notebook_config.py
[[ -f "$f" ]] || jupyter notebook --generate-config

l="c.ContentsManager.line_numbers = True"
grep "$l" $f >/dev/null || echo "$l" >>$f


installing jupyter...


## 步骤8: 安装常用软件


In [ ]:
%%bash
echo "installing google-chrome..."
yay -S --needed --noconfirm google-chrome >/dev/null

echo "installing keepassxc..."
sudo pacman -S --needed --noconfirm keepassxc >/dev/null

echo "installing cryptomator..."
yay -S --needed --noconfirm cryptomator-bin >/dev/null


## 步骤9: 配置 pacman-key


In [7]:
%%bash
echo "installing pacman-key..."
sudo pacman -S --needed --noconfirm manjaro-keyring archlinux-keyring archlinuxcn-keyring >/dev/null
sudo pacman-key --init >/dev/null
sudo pacman-key --populate archlinux manjaro archlinuxcn >/dev/null
sudo pacman -Syy --noconfirm >/dev/null


installing pacman-key...


gpg: public key CF66D153D884358F is 16 seconds newer than the signature
gpg: error reading key: No public key
gpg: error reading key: No public key
gpg: changing ownertrust from 132 to 4
gpg: Note: third-party key signatures using the SHA1 algorithm are rejected
gpg: (use option "--allow-weak-key-signatures" to override)
gpg: marginals needed: 3  completes needed: 1  trust model: pgp
gpg: public key CF66D153D884358F is 16 seconds newer than the signature
gpg: depth: 0  valid:   1  signed:  69  trust: 0-, 0q, 0n, 0m, 0f, 1u
gpg: depth: 1  valid:  69  signed: 100  trust: 0-, 0q, 0n, 69m, 0f, 0u
gpg: depth: 2  valid:  75  signed:  20  trust: 75-, 0q, 0n, 0m, 0f, 0u
gpg: next trustdb check due at 2026-01-30


## 步骤10: 安装代理工具


In [ ]:
%%bash
echo "installing clash-verge..."
sudo pacman -S --needed --noconfirm clash-verge-rev >/dev/null


## 步骤11: 安装和配置输入法


In [ ]:
%%bash
echo "installing fcitx5..."
sudo pacman -S --needed --noconfirm \
	fcitx5 \
	fcitx5-gtk \
	fcitx5-qt \
	fcitx5-configtool \
	fcitx5-chinese-addons \
	fcitx5-pinyin-zhwiki >/dev/null
kwriteconfig6 --file kwinrc --group Wayland --key 'InputMethod' /usr/share/applications/org.fcitx.Fcitx5.desktop


## 步骤12: 配置 git 和 ssh


### 配置 Git


In [ ]:
%%bash
name="kefu"
email="19157521820@163.com"
signingkey=$(gpg --list-secret-keys --keyid-format SHORT 2>/dev/null | grep sec | awk '{print $2}' | cut -d'/' -f2 | head -1)

git config --global user.name "$name"
git config --global user.email "$email"
git config --global init.defaultBranch "main"
git config --global gpg.program "gpg"
git config --global user.signingkey "$signingkey"
git config --global commit.gpgsign "false"
git config --global credential.helper "store"

echo "✓ Git configured"


### 配置 SSH


In [ ]:
%%bash
email="19157521820@163.com"
ssh_key_type="ed25519"
ssh_dir="/data/.home/.ssh"

mkdir -p "$ssh_dir"
ssh_real_path=$(readlink -f "$ssh_dir")
ssh_key_path="$ssh_real_path/id_$ssh_key_type"

if [[ ! -f "$ssh_key_path" ]]; then
	ssh-keygen -t "$ssh_key_type" -C "$email" -f "$ssh_key_path" -N ""
	echo "✓ SSH key generated"
else
	echo "✓ SSH key already exists"
fi

chmod 700 "$ssh_real_path"
chmod 600 "$ssh_real_path"/id_* 2>/dev/null || true
chmod 644 "$ssh_real_path"/*.pub 2>/dev/null || true
[[ -f "$ssh_real_path/config" ]] && chmod 600 "$ssh_real_path/config"

ssh_home_path="$HOME/.ssh"
rm -rf "$ssh_home_path"
ln -sf "$ssh_real_path" "$ssh_home_path"


### 配置自动启动


In [ ]:
%%bash
autostart_dir="$HOME/.config/autostart"
mkdir -p "$autostart_dir"

app="cryptomator"
exec_path=$(which "$app" 2>/dev/null)

if [[ -n "$exec_path" ]]; then
	cat > "$autostart_dir/$app.desktop" <<EOF
[Desktop Entry]
Type=Application
Name=$app
Exec=$exec_path
Icon=$app
Comment=Auto-start $app
X-GNOME-Autostart-enabled=true
StartupNotify=false
Terminal=false
EOF
	echo "Created autostart file for $app"
else
	echo "Failed to find $app"
fi


## 步骤13: 启动应用程序


In [ ]:
%%bash
cryptomator >/dev/null &
keepassxc >/dev/null &
code . >/dev/null &
clash-verge >/dev/null &
i=$(ip addr show | grep -E 'inet.*global' | awk '{print $2}' | cut -d'/' -f1 | head -n1) && echo "Using IP: $i "
google-chrome-stable --proxy-server="socks5://${i}:7897" >/dev/null &
echo "Done!!!"
